In [1]:
import pandas as pd 


In [3]:
df = pd.read_csv("credit_risk_raw.csv")

print("Rows and Columns:", df.shape)
df.head(10)

Rows and Columns: (3060, 13)


,loan_id,application_date,annual_income,loan_amount,credit_score,debt_to_income,employment_years,home_ownership,loan_purpose,prior_delinquencies,revolving_utilization,term_months,default_12m
0,L002805,2024-04-27,93546.15,46456.08,712.0,0.564,6,rent,education,3.0,0.409,36,1.0
1,L002049,2024-03-13,50788.43,8995.86,711.0,0.499,15,own,education,2.0,0.167,36,0.0
2,L002311,2022-08-27,105239.24,8045.67,754.0,0.49,14,rent,education,1.0,0.617,60,0.0
3,L002086,2020-07-20,46277.34,53654.7,737.0,0.464,23,rent,debt_consolidation,0.0,0.659,60,0.0
4,L002925,2021-01-05,66339.78,10459,754.0,0.22,10,MORTGAGE,personal,0.0,0.027,36,0.0
5,L000584,2021-03-24,40542.88,40799.47,574.0,0.692,20,RENT,personal,0.0,0.39,60,1.0
6,L001418,2019-11-05,36389.47,45017.93,650.0,0.687,4,own,home_improvement,0.0,0.501,60,1.0
7,L001854,2022-05-24,58507.34,59857.83,674.0,0.259,13,mortgage,education,1.0,0.932,60,0.0
8,L000190,2021-08-06,108286.94,39499.77,784.0,0.073,10,RENT,personal,2.0,0.985,36,0.0
9,L002991,2020-02-08,42672.17,58053.84,660.0,0.07,3,own,education,1.0,0.472,60,1.0


In [4]:
df.info()

print("\nMissing Values:")
print(df.isna().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   loan_id                3060 non-null   object 
 1   application_date       3045 non-null   object 
 2   annual_income          2979 non-null   object 
 3   loan_amount            3040 non-null   object 
 4   credit_score           2968 non-null   float64
 5   debt_to_income         3010 non-null   object 
 6   employment_years       2989 non-null   object 
 7   home_ownership         3019 non-null   object 
 8   loan_purpose           3034 non-null   object 
 9   prior_delinquencies    3035 non-null   float64
 10  revolving_utilization  3030 non-null   object 
 11  term_months            3060 non-null   object 
 12  default_12m            3029 non-null   float64
dtypes: float64(3), object(10)
memory usage: 310.9+ KB

Missing Values:
loan_id                   0
application_d

In [5]:
df_raw = df.copy()

df = df.drop_duplicates().reset_index(drop=True)

print("Original rows:", len(df_raw))
print("Rows after removing duplicates:", len(df))
print("Remaining exact duplicates:", df.duplicated().sum())

Original rows: 3060
Rows after removing duplicates: 3000
Remaining exact duplicates: 0


In [7]:
income_check = pd.to_numeric(df["annual_income"], errors="coerce")

df.loc[
    income_check.isna(),
    ["loan_id", "annual_income"]
].head(15)

,loan_id,annual_income
39,L002939,"$49,892.04"
56,L002032,unknown
70,L001831,NaN
124,L001719,NaN
131,L001141,NaN
151,L000182,NaN
153,L001292,unknown
159,L001513,unknown
177,L002345,"$133,843.46"
178,L002835,NaN


In [8]:
income_text = df["annual_income"].astype("string").str.strip()

income_text = income_text.str.replace("$", "", regex=False)
income_text = income_text.str.replace(",", "", regex=False)

df["annual_income"] = pd.to_numeric(income_text, errors="coerce")

print("Data type:", df["annual_income"].dtype)
print("Missing incomes:", df["annual_income"].isna().sum())

df.loc[
    income_check.isna(),
    ["loan_id", "annual_income"]
].head(15)


Data type: Float64
Missing incomes: 120


,loan_id,annual_income
39,L002939,49892.04
56,L002032,<NA>
70,L001831,<NA>
124,L001719,<NA>
131,L001141,<NA>
151,L000182,<NA>
153,L001292,<NA>
159,L001513,<NA>
177,L002345,133843.46
178,L002835,<NA>


In [9]:
invalid_income = df["annual_income"] <= 0

print("Number of invalid income:", invalid_income.sum())

df.loc[
    invalid_income,
    ["loan_id", "annual_income"]
]

Number of invalid income: 10


,loan_id,annual_income
29,L000958,0.0
37,L000935,-45000.0
492,L001477,-45000.0
778,L002265,-45000.0
1428,L000528,-45000.0
1646,L000367,0.0
1665,L001084,0.0
1924,L001724,0.0
2631,L002790,-45000.0
2835,L000198,0.0


In [10]:
# Replace invalid income values with a missing-value marker
df.loc[invalid_income, "annual_income"] = pd.NA

# Verify the change
print("Missing incomes:", df["annual_income"].isna().sum())
print("Remaining invalid incomes:", (df["annual_income"] <= 0).sum())
print("Total rows:", len(df))

Missing incomes: 130
Remaining invalid incomes: 0
Total rows: 3000


In [14]:
loan_check = pd.to_numeric(df["loan_amount"], errors="coerce")

problem_loans = loan_check.isna() | (loan_check <= 0)

df.loc[
    problem_loans,
    ["loan_id", "loan_amount"]
].head(15)



,loan_id,loan_amount
11,L001753,"35,137.73"
12,L002482,"58,606.78"
76,L001351,"28,461.98"
80,L002969,"14,054.06"
173,L000013,"51,576.23"
192,L002419,"39,818.60"
210,L000908,"45,145.95"
214,L000455,NaN
249,L002951,NaN
263,L001621,"12,731.33"


In [15]:
loan_text = df["loan_amount"].astype("string").str.strip()
loan_text =  loan_text.str.replace(",", "", regex=False)

df["loan_amount"] = pd.to_numeric(loan_text, errors="coerce")

df.loc[df["loan_amount"] <= 0, "loan_amount"] = pd.NA

print("Data type:", df["loan_amount"].dtype)
print("Missing loan amounts:", df["loan_amount"].isna().sum())
print("Remaining invalid amounts:", (df["loan_amount"] <= 0).sum())

Data type: Float64
Missing loan amounts: 30
Remaining invalid amounts: 0


In [16]:
invalid_credit = (
    (df["credit_score"] < 300) | 
    (df["credit_score"] > 850)
)

print("Missing credit score:", df["credit_score"].isna().sum())
print("Invalid credit scores:", invalid_credit.sum())

# Display the invalid records
df.loc[
    invalid_credit,
    ["loan_id", "credit_score"]
].head(10)

Missing credit score: 90
Invalid credit scores: 15


,loan_id,credit_score
57,L000771,950.0
362,L002501,-100.0
523,L000327,950.0
1220,L002583,-100.0
1366,L000023,950.0
1430,L002762,-100.0
1998,L000030,-100.0
2138,L001432,-100.0
2610,L002878,950.0
2643,L002649,-100.0


In [17]:
df.loc[invalid_credit, "credit_score"] = pd.NA

remaining_invalid_credit = (
    (df["credit_score"] < 300) | 
    (df["credit_score"] > 850)
)

print("Missing credit score:", df["credit_score"].isna().sum())
print("Remaining invalid scores:", remaining_invalid_credit.sum())

Missing credit score: 105
Remaining invalid scores: 0


In [18]:
# Try interpreting the raw values as numbers
dti_check = pd.to_numeric(df["debt_to_income"], errors="coerce")

# Identify missing, unparseable, or out-of-range values
problem_dti = (
    dti_check.isna() |
    (dti_check < 0) |
    (dti_check > 1)
)

df.loc[
    problem_dti,
    ["loan_id", "debt_to_income"]
].head(15)

,loan_id,debt_to_income
47,L000769,-0.25
64,L001161,10.1%
83,L001248,NaN
84,L002764,17.9%
98,L002134,25.8%
114,L001797,19.2%
137,L001586,NaN
152,L002608,39.5%
176,L002150,18.8%
195,L001437,57.5%


In [19]:
# Prepare the text and remember which values contain %
dti_text = df["debt_to_income"].astype("string").str.strip()
is_percent = dti_text.str.endswith("%", na=False)

# Remove % and convert to numbers
dti_numbers = pd.to_numeric(
    dti_text.str.replace("%", "", regex=False),
    errors="coerce"
)

# Divide only the original percentage values by 100
dti_numbers.loc[is_percent] = dti_numbers.loc[is_percent] / 100

# Mark values outside this dataset's allowed range as missing
invalid_dti = (dti_numbers < 0) | (dti_numbers > 1)
dti_numbers.loc[invalid_dti] = pd.NA

# Save the cleaned column
df["debt_to_income"] = dti_numbers

# Verify the result
print("Missing DTI values:", df["debt_to_income"].isna().sum())
print(
    "Remaining invalid DTI values:",
    ((df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)).sum()
)

# Show the previously flagged rows
df.loc[problem_dti, ["loan_id", "debt_to_income"]].head(15)

Missing DTI values: 60
Remaining invalid DTI values: 0


,loan_id,debt_to_income
47,L000769,<NA>
64,L001161,0.101
83,L001248,<NA>
84,L002764,0.179
98,L002134,0.258
114,L001797,0.192
137,L001586,<NA>
152,L002608,0.395
176,L002150,0.188
195,L001437,0.575


In [21]:
employment_check = pd.to_numeric(
    df["employment_years"], 
    errors="coerce"
)

problem_employee = (
    employment_check.isna() | 
    (employment_check < 0) | 
    (employment_check > 25)
)

df.loc[
    problem_employee,
    ["loan_id", "employment_years"]
].head(15)

,loan_id,employment_years
31,L002197,2 years
51,L001310,NaN
55,L000799,NaN
79,L000666,7 years
88,L001734,NaN
116,L001040,24 years
162,L001325,-5
202,L001606,2 years
204,L000271,8 years
265,L000087,NaN


In [22]:
# Standardize the text and remove the unit
employment_text = df["employment_years"].astype("string").str.lower()
employment_text = employment_text.str.replace("years", "", regex=False)
employment_text = employment_text.str.strip()

# Convert to numbers
df["employment_years"] = pd.to_numeric(
    employment_text,
    errors="coerce"
)

# Mark values outside 0–25 as missing
invalid_employment = (
    (df["employment_years"] < 0) |
    (df["employment_years"] > 25)
)
df.loc[invalid_employment, "employment_years"] = pd.NA

# Verify the result
print("Data type:", df["employment_years"].dtype)
print("Missing employment years:", df["employment_years"].isna().sum())
print(
    "Remaining invalid values:",
    ((df["employment_years"] < 0) | (df["employment_years"] > 25)).sum()
)

Data type: Int64
Missing employment years: 80
Remaining invalid values: 0


In [23]:
# Show the different labels exactly as stored
print(df["home_ownership"].unique())

# Count each label, including missing values
df["home_ownership"].value_counts(dropna=False)

['rent' 'own' ' MORTGAGE ' ' RENT ' 'mortgage' ' OWN ' nan]


home_ownership
mortgage      970
own           968
rent          922
 RENT          42
NaN            40
 OWN           30
 MORTGAGE      28
Name: count, dtype: int64

In [24]:
df["home_ownership"] = (
    df["home_ownership"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Check the standardized categories
df["home_ownership"].value_counts(dropna=False)

home_ownership
own         998
mortgage    998
rent        964
<NA>         40
Name: count, dtype: Int64

In [25]:
# Show the labels exactly as stored
print(df["loan_purpose"].unique())

# Count each label, including missing values
df["loan_purpose"].value_counts(dropna=False)

['education' 'debt_consolidation' 'personal' 'home_improvement' 'medical'
 ' PERSONAL ' ' MEDICAL ' nan ' HOME IMPROVEMENT ' ' EDUCATION '
 ' DEBT CONSOLIDATION ']


loan_purpose
debt_consolidation      593
home_improvement        590
medical                 582
personal                570
education               565
NaN                      25
 EDUCATION               23
 HOME IMPROVEMENT        16
 MEDICAL                 14
 PERSONAL                11
 DEBT CONSOLIDATION      11
Name: count, dtype: int64

In [26]:
df["loan_purpose"] = (
    df["loan_purpose"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Check the standardized categories
df["loan_purpose"].value_counts(dropna=False)

loan_purpose
home_improvement      606
debt_consolidation    604
medical               596
education             588
personal              581
<NA>                   25
Name: count, dtype: Int64

In [27]:
df["prior_delinquencies"].value_counts(dropna=False)

prior_delinquencies
 0.0    1459
 1.0     780
 2.0     372
 3.0     354
 NaN      25
-2.0      10
Name: count, dtype: int64

In [28]:
# Mark negative counts as missing
df.loc[
    df["prior_delinquencies"] < 0,
    "prior_delinquencies"
] = pd.NA

# Store whole-number counts while allowing missing values
df["prior_delinquencies"] = df["prior_delinquencies"].astype("Int64")

# Verify the result
print("Data type:", df["prior_delinquencies"].dtype)
print("Missing counts:", df["prior_delinquencies"].isna().sum())
print("Remaining negative counts:", (df["prior_delinquencies"] < 0).sum())

Data type: Int64
Missing counts: 35
Remaining negative counts: 0


In [29]:
utilization_check = pd.to_numeric(
    df["revolving_utilization"],
    errors="coerce"
)

problem_utilization = (
    utilization_check.isna() |
    (utilization_check < 0) |
    (utilization_check > 1)
)

df.loc[
    problem_utilization,
    ["loan_id", "revolving_utilization"]
].head(15)

,loan_id,revolving_utilization
10,L000645,-0.4
14,L001564,40.7%
54,L002363,39.8%
96,L000052,79.3%
275,L001428,13.9%
291,L002331,47.3%
293,L001427,-0.4
304,L000134,90.8%
388,L000067,94.9%
393,L001867,NaN


In [30]:
# 1. Convert to text and remove surrounding spaces
utilization_text = df["revolving_utilization"].astype("string").str.strip()

# 2. Identify percentage entries before removing "%"
is_percent = utilization_text.str.endswith("%", na=False)

# 3. Remove "%" and convert to numbers
utilization_numbers = pd.to_numeric(
    utilization_text.str.replace("%", "", regex=False),
    errors="coerce"
)

# 4. Convert only percentage entries to decimals
utilization_numbers.loc[is_percent] = (
    utilization_numbers.loc[is_percent] / 100
)

# 5. Replace values outside the expected range with missing values
invalid_utilization = (
    (utilization_numbers < 0) |
    (utilization_numbers > 1)
)
utilization_numbers.loc[invalid_utilization] = pd.NA

# 6. Update the column
df["revolving_utilization"] = utilization_numbers

# 7. Check the result
print("Missing utilization:", df["revolving_utilization"].isna().sum())
print(
    "Remaining invalid values:",
    (
        (df["revolving_utilization"] < 0) |
        (df["revolving_utilization"] > 1)
    ).sum()
)

Missing utilization: 40
Remaining invalid values: 0


In [31]:
print(df["term_months"].value_counts(dropna=False))

term_months
60           1504
36           1456
60 months      15
36 months      15
0              10
Name: count, dtype: int64


In [32]:
# Remove the unit and surrounding spaces
term_text = (
    df["term_months"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("months", "", regex=False)
    .str.strip()
)

# Convert to numbers
term_numbers = pd.to_numeric(term_text, errors="coerce")

# Keep only the allowed loan terms
invalid_terms = ~term_numbers.isin([36, 60])
term_numbers.loc[invalid_terms] = pd.NA

# Store as whole numbers while allowing missing values
df["term_months"] = term_numbers.astype("Int64")

# Check the result
print(df["term_months"].value_counts(dropna=False))
print("Missing terms:", df["term_months"].isna().sum())

term_months
60      1519
36      1471
<NA>      10
Name: count, dtype: Int64
Missing terms: 10


In [33]:
# Preview the original date values
print(df["application_date"].head(10))

# Check which entries match the YYYY-MM-DD format
date_check = pd.to_datetime(
    df["application_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Show entries that did not match, including missing dates
df.loc[
    date_check.isna(),
    ["loan_id", "application_date"]
].head(20)

0    2024-04-27
1    2024-03-13
2    2022-08-27
3    2020-07-20
4    2021-01-05
5    2021-03-24
6    2019-11-05
7    2022-05-24
8    2021-08-06
9    2020-02-08
Name: application_date, dtype: object


,loan_id,application_date
25,L002267,2/11/2019
66,L001212,11/24/2020
86,L002145,1/6/2023
107,L000373,7/23/2023
135,L002897,5/30/2020
148,L002813,3/10/2020
156,L001185,8/2/2024
169,L002948,2024-13-01
172,L003000,2024-13-01
184,L001832,1/2/2019


In [34]:
# Remove surrounding spaces
date_text = df["application_date"].astype("string").str.strip()

# Parse dates written as YYYY-MM-DD
dates_iso = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

# Parse dates written as MM/DD/YYYY
dates_us = pd.to_datetime(
    date_text,
    format="%m/%d/%Y",
    errors="coerce"
)

# Use the second result wherever the first result is missing
df["application_date"] = dates_iso.combine_first(dates_us)

# Check the cleaned column
print("Data type:", df["application_date"].dtype)
print("Missing dates:", df["application_date"].isna().sum())
print("Earliest date:", df["application_date"].min())
print("Latest date:", df["application_date"].max())

Data type: datetime64[ns]
Missing dates: 25
Earliest date: 2019-01-01 00:00:00
Latest date: 2024-12-31 00:00:00


In [35]:
df["application_date"].head(10)

0   2024-04-27
1   2024-03-13
2   2022-08-27
3   2020-07-20
4   2021-01-05
5   2021-03-24
6   2019-11-05
7   2022-05-24
8   2021-08-06
9   2020-02-08
Name: application_date, dtype: datetime64[ns]

In [36]:
print(df["default_12m"].value_counts(dropna=False))
print("Data type:", df["default_12m"].dtype)

default_12m
0.0    2313
1.0     657
NaN      30
Name: count, dtype: int64
Data type: float64


In [37]:
df["default_12m"] = df["default_12m"].astype("Int64")

print(df["default_12m"].value_counts(dropna=False))
print("Data type:", df["default_12m"].dtype)

default_12m
0       2313
1        657
<NA>      30
Name: count, dtype: Int64
Data type: Int64


In [38]:
print("Total rows:", len(df))
print("\nMissing values by column:")
print(df.isna().sum())

Total rows: 3000

Missing values by column:
loan_id                    0
application_date          25
annual_income            130
loan_amount               30
credit_score             105
debt_to_income            60
employment_years          80
home_ownership            40
loan_purpose              25
prior_delinquencies       35
revolving_utilization     40
term_months               10
default_12m               30
dtype: int64


In [39]:
df_model =  df.dropna(subset=["default_12m", "application_date"]).copy()

print("Cleaned datasets rows:", len(df))
print("Modeling datasets rows:", len(df_model))
print("Excluded rows:", len(df) - len(df_model))

print(
    "\nMissing dates or targets:",
    df_model[["application_date", "default_12m"]].isna().sum()
)

Cleaned datasets rows: 3000
Modeling datasets rows: 2945
Excluded rows: 55

Missing dates or targets: application_date    0
default_12m         0
dtype: int64


In [40]:
# Put applications in chronological order
df_model = df_model.sort_values("application_date").reset_index(drop=True)

# Split using fixed date boundaries
train_df = df_model.loc[
    df_model["application_date"] < "2023-01-01"
].copy()

gap_df = df_model.loc[
    (df_model["application_date"] >= "2023-01-01") &
    (df_model["application_date"] < "2024-01-01")
].copy()

test_df = df_model.loc[
    df_model["application_date"] >= "2024-01-01"
].copy()

# Check the sizes and date ranges
print("Training rows:", len(train_df))
print("Gap rows:", len(gap_df))
print("Test rows:", len(test_df))

print(
    "\nTraining dates:",
    train_df["application_date"].min(),
    "to",
    train_df["application_date"].max()
)
print(
    "Test dates:",
    test_df["application_date"].min(),
    "to",
    test_df["application_date"].max()
)

Training rows: 1949
Gap rows: 518
Test rows: 478

Training dates: 2019-01-01 00:00:00 to 2022-12-30 00:00:00
Test dates: 2024-01-02 00:00:00 to 2024-12-31 00:00:00


In [41]:
# Count each outcome in the training data
counts = train_df["default_12m"].value_counts().sort_index()

# Calculate the percentage of each outcome
percentages = (counts / len(train_df) * 100).round(2)

# Display both together
target_summary = pd.DataFrame({
    "Count": counts,
    "Percentage (%)": percentages
})

target_summary.index.name = "default_12m"
display(target_summary)

,Count,Percentage (%)
default_12m,,
0,1519,77.94
1,430,22.06


In [42]:
credit_score_summary = (
    train_df
    .groupby("default_12m")["credit_score"]
    .agg(["count", "mean", "median"])
    .round(2)
)

display(credit_score_summary)

,count,mean,median
default_12m,,,
0,1471,687.71,686.0
1,407,631.53,627.0


In [43]:
dti_summary = (
    train_df
    .groupby("default_12m")["debt_to_income"]
    .agg(["count", "mean", "median"])
    .round(3)
)

display(dti_summary)

,count,mean,median
default_12m,,,
0,1487,0.35,0.341
1,424,0.465,0.498


In [44]:
delinquency_summary = (
    train_df
    .groupby("prior_delinquencies", dropna=False)["default_12m"]
    .agg(
        applications="count",
        defaults="sum",
        default_rate_percent="mean"
    )
)

# Convert the default rate to a percentage
delinquency_summary["default_rate_percent"] = (
    delinquency_summary["default_rate_percent"] * 100
).round(2)

display(delinquency_summary)

,applications,defaults,default_rate_percent
prior_delinquencies,,,
0,918,147,16.01
1,526,114,21.67
2,248,67,27.02
3,236,96,40.68
<NA>,21,6,28.57


In [46]:
feature_columns = train_df.columns.drop(
    ["loan_id", "application_date", "default_12m"]
).tolist()

X_train = train_df[feature_columns].copy()
y_train = train_df["default_12m"].astype(int)

X_test = test_df[feature_columns].copy()
y_test = test_df["default_12m"].astype(int)

# Check dimensions
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1949, 10)
y_train: (1949,)
X_test: (478, 10)
y_test: (478,)


In [47]:
numeric_features = [
    "annual_income",
    "loan_amount",
    "credit_score",
    "debt_to_income",
    "employment_years",
    "prior_delinquencies",
    "revolving_utilization",
    "term_months"
]

categorical_features = [
    "home_ownership",
    "loan_purpose"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 8
Categorical features: 2


In [53]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Use compatible data types and missing-value markers for sklearn
for inputs in (X_train, X_test):
    inputs[numeric_features] = inputs[numeric_features].astype("float64")

    inputs[categorical_features] = (
        inputs[categorical_features].astype(object)
    )
    inputs[categorical_features] = inputs[categorical_features].where(
        inputs[categorical_features].notna(),
        np.nan
    )

# Steps for numeric columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Steps for categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="missing"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply each pipeline to its corresponding columns
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    sparse_threshold=0
)

print("Preprocessing pipeline ready.")

Preprocessing pipeline ready.


In [54]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression 

logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

print("Logistic regression trained.")

Logistic regression trained.


In [56]:
cv_splits = []
training_dates = train_df["application_date"]

for validation_year in [2021, 2022]:
    # Training outcomes must be known before validation begins
    train_cutoff = pd.Timestamp(f"{validation_year - 1}-01-01")
    validation_start = pd.Timestamp(f"{validation_year}-01-01")
    validation_end = pd.Timestamp(f"{validation_year + 1}-01-01")

    # Row positions for this round
    train_positions = np.flatnonzero(
        (training_dates < train_cutoff).to_numpy()
    )

    validation_positions = np.flatnonzero(
        (
            (training_dates >= validation_start) &
            (training_dates < validation_end)
        ).to_numpy()
    )

    cv_splits.append((train_positions, validation_positions))

    print(
        f"Validation year {validation_year}: "
        f"{len(train_positions)} training rows, "
        f"{len(validation_positions)} validation rows"
    )

Validation year 2021: 493 training rows, 472 validation rows
Validation year 2022: 979 training rows, 498 validation rows


In [57]:
from sklearn.model_selection import cross_validate

# Refit and evaluate the full pipeline in each validation round
lr_cv_results = cross_validate(
    estimator=logistic_model,
    X=X_train,
    y=y_train,
    cv=cv_splits,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision"
    },
    error_score="raise"
)

# Display the results for each year
lr_cv_summary = pd.DataFrame({
    "Validation year": [2021, 2022],
    "ROC AUC": lr_cv_results["test_roc_auc"],
    "Average precision": lr_cv_results["test_average_precision"]
})

display(lr_cv_summary.round(3))

print(
    "Mean validation ROC AUC:",
    round(lr_cv_results["test_roc_auc"].mean(), 3)
)
print(
    "Mean validation average precision:",
    round(lr_cv_results["test_average_precision"].mean(), 3)
)

,Validation year,ROC AUC,Average precision
0,2021,0.845,0.625
1,2022,0.790,0.580


Mean validation ROC AUC: 0.817
Mean validation average precision: 0.603


In [58]:
from sklearn.ensemble import GradientBoostingClassifier

# Combine preprocessing and gradient boosting
gb_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ))
])

# Train using the same training dataset
gb_model.fit(X_train, y_train)

print("Gradient boosting trained.")

Gradient boosting trained.


In [59]:
# Validate gradient boosting
gb_cv_results = cross_validate(
    estimator=gb_model,
    X=X_train,
    y=y_train,
    cv=cv_splits,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision"
    },
    error_score="raise"
)

# Results for each validation year
gb_cv_summary = pd.DataFrame({
    "Validation year": [2021, 2022],
    "ROC AUC": gb_cv_results["test_roc_auc"],
    "Average precision": gb_cv_results["test_average_precision"]
})

display(gb_cv_summary.round(3))

# Compare the mean validation scores
model_comparison = pd.DataFrame({
    "Model": ["Logistic regression", "Gradient boosting"],
    "Mean ROC AUC": [
        lr_cv_results["test_roc_auc"].mean(),
        gb_cv_results["test_roc_auc"].mean()
    ],
    "Mean average precision": [
        lr_cv_results["test_average_precision"].mean(),
        gb_cv_results["test_average_precision"].mean()
    ]
})

display(model_comparison.round(3))

,Validation year,ROC AUC,Average precision
0,2021,0.816,0.621
1,2022,0.753,0.571


,Model,Mean ROC AUC,Mean average precision
0,Logistic regression,0.817,0.603
1,Gradient boosting,0.784,0.596


In [60]:
from sklearn.model_selection import GridSearchCV

# 2 × 2 × 2 = 8 combinations
gb_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [1, 2]
}

gb_search = GridSearchCV(
    estimator=clone(gb_model),
    param_grid=gb_param_grid,
    cv=cv_splits,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision"
    },
    refit="roc_auc",
    error_score="raise"
)

gb_search.fit(X_train, y_train)

# Results for the selected combination
best_index = gb_search.best_index_

print("Best settings:", gb_search.best_params_)
print("Best mean validation ROC AUC:", round(gb_search.best_score_, 3))
print(
    "Corresponding mean average precision:",
    round(
        gb_search.cv_results_[
            "mean_test_average_precision"
        ][best_index],
        3
    )
)

Best settings: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 1, 'classifier__n_estimators': 200}
Best mean validation ROC AUC: 0.801
Corresponding mean average precision: 0.595


In [61]:
lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0]
}

lr_search = GridSearchCV(
    estimator=clone(logistic_model),
    param_grid=lr_param_grid,
    cv=cv_splits,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision"
    },
    refit="roc_auc",
    error_score="raise"
)

lr_search.fit(X_train, y_train)

# Results for the selected combination
best_index = lr_search.best_index_

print("Best settings:", lr_search.best_params_)
print("Best mean validation ROC AUC:", round(lr_search.best_score_, 3))
print(
    "Corresponding mean average precision:",
    round(
        lr_search.cv_results_[
            "mean_test_average_precision"
        ][best_index],
        3
    )
)

Best settings: {'classifier__C': 0.1}
Best mean validation ROC AUC: 0.818
Corresponding mean average precision: 0.606


In [62]:
# GridSearchCV already refitted this pipeline on all training rows
final_model = lr_search.best_estimator_
final_model_name = "Logistic regression"

validation_parts = []

# Generate predictions using only earlier rows in each round
for year, (fit_positions, validation_positions) in zip(
    [2021, 2022], cv_splits
):
    fold_model = clone(final_model)

    fold_model.fit(
        X_train.iloc[fit_positions],
        y_train.iloc[fit_positions]
    )

    probabilities = fold_model.predict_proba(
        X_train.iloc[validation_positions]
    )[:, 1]

    validation_parts.append(pd.DataFrame({
        "validation_year": year,
        "actual_default": y_train.iloc[
            validation_positions
        ].to_numpy(),
        "default_probability": probabilities
    }))

validation_predictions = pd.concat(
    validation_parts,
    ignore_index=True
)

print("Selected model:", final_model_name)
print("Validation predictions:", len(validation_predictions))
display(validation_predictions.head())

Selected model: Logistic regression
Validation predictions: 970


,validation_year,actual_default,default_probability
0,2021,1,0.593395
1,2021,1,0.155630
2,2021,1,0.323499
3,2021,0,0.332381
4,2021,0,0.014891


In [63]:
from sklearn.metrics import precision_score, recall_score

actual = validation_predictions["actual_default"]
probabilities = validation_predictions["default_probability"]

threshold_rows = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    review = (probabilities >= threshold).astype(int)

    missed_defaults = ((actual == 1) & (review == 0)).sum()

    threshold_rows.append({
        "Threshold": threshold,
        "Review count": int(review.sum()),
        "Review (%)": review.mean() * 100,
        "Precision (%)": precision_score(
            actual, review, zero_division=0
        ) * 100,
        "Recall (%)": recall_score(
            actual, review, zero_division=0
        ) * 100,
        "Missed defaults": int(missed_defaults)
    })

threshold_summary = pd.DataFrame(threshold_rows)

display(threshold_summary.round(2))

,Threshold,Review count,Review (%),Precision (%),Recall (%),Missed defaults
0,0.10,640,65.98,30.31,91.94,17
1,0.15,504,51.96,36.31,86.73,28
2,0.20,405,41.75,40.99,78.67,45
3,0.25,331,34.12,44.71,70.14,63
4,0.30,261,26.91,50.96,63.03,78
5,0.40,161,16.60,62.73,47.87,110
6,0.50,103,10.62,68.93,33.65,140


In [64]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Fix the threshold selected from validation
review_threshold = 0.15

# Predict default probabilities for the reserved test data
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply the review rule
test_review = (
    test_probabilities >= review_threshold
).astype(int)

# Calculate test performance
test_metrics = {
    "ROC AUC": roc_auc_score(y_test, test_probabilities),
    "Average precision": average_precision_score(
        y_test, test_probabilities
    ),
    "Precision (%)": precision_score(
        y_test, test_review, zero_division=0
    ) * 100,
    "Recall (%)": recall_score(
        y_test, test_review, zero_division=0
    ) * 100,
    "Review count": int(test_review.sum()),
    "Review (%)": test_review.mean() * 100
}

display(
    pd.DataFrame.from_dict(
        test_metrics,
        orient="index",
        columns=["Test result"]
    ).round(3)
)

,Test result
ROC AUC,0.810
Average precision,0.620
Precision (%),37.500
Recall (%),83.898
Review count,264.000
Review (%),55.230


In [65]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    test_review,
    labels=[0, 1]
)

confusion_table = pd.DataFrame(
    cm,
    index=["Actual non-default", "Actual default"],
    columns=["Not flagged", "Flagged for review"]
)

display(confusion_table)

,Not flagged,Flagged for review
Actual non-default,195,165
Actual default,19,99


In [66]:
# Keep predictions and actual outcomes for later evaluation
test_predictions = test_df[
    ["loan_id", "application_date"]
].copy()

test_predictions["default_probability"] = test_probabilities
test_predictions["flagged_for_review"] = test_review
test_predictions["actual_default"] = y_test.to_numpy()

# Build the queue without actual outcomes
review_queue = (
    test_predictions.loc[
        test_predictions["flagged_for_review"] == 1,
        ["loan_id", "application_date", "default_probability"]
    ]
    .sort_values("default_probability", ascending=False)
    .reset_index(drop=True)
)

# Assign review priority
review_queue.insert(
    0,
    "review_priority",
    np.arange(1, len(review_queue) + 1)
)

print("Applications in review queue:", len(review_queue))
display(
    review_queue.head(10).round({"default_probability": 3})
)

Applications in review queue: 264


,review_priority,loan_id,application_date,default_probability
0,1,L002516,2024-11-12,0.913
1,2,L000694,2024-12-30,0.886
2,3,L001222,2024-06-08,0.864
3,4,L000600,2024-11-28,0.853
4,5,L002692,2024-05-08,0.840
5,6,L000855,2024-09-14,0.837
6,7,L002312,2024-03-10,0.837
7,8,L001783,2024-11-13,0.828
8,9,L000451,2024-01-17,0.816
9,10,L002518,2024-05-07,0.795


In [67]:
# Get the names and weights of the processed features
processed_names = (
    final_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    final_model.named_steps["classifier"].coef_[0]
)

feature_effects = pd.DataFrame({
    "Feature": processed_names,
    "Coefficient": coefficients
})

# Inspect numeric features first
numeric_effects = feature_effects.loc[
    feature_effects["Feature"].str.startswith("numeric__")
].copy()

numeric_effects["Absolute coefficient"] = (
    numeric_effects["Coefficient"].abs()
)

numeric_effects = numeric_effects.sort_values(
    "Absolute coefficient",
    ascending=False
)

display(numeric_effects.round(3))

,Feature,Coefficient,Absolute coefficient
2,numeric__credit_score,-1.008,1.008
3,numeric__debt_to_income,0.766,0.766
5,numeric__prior_delinquencies,0.537,0.537
1,numeric__loan_amount,0.270,0.270
0,numeric__annual_income,-0.242,0.242
6,numeric__revolving_utilization,0.159,0.159
4,numeric__employment_years,-0.134,0.134
7,numeric__term_months,0.045,0.045


In [68]:
from pathlib import Path
import joblib
import sklearn

# Create an output folder
output_dir = Path("credit_risk_outputs")
output_dir.mkdir(exist_ok=True)

# Save the cleaned data and results
df.to_csv(output_dir / "cleaned_data.csv", index=False)
test_predictions.to_csv(
    output_dir / "test_predictions.csv", index=False
)
review_queue.to_csv(
    output_dir / "review_queue.csv", index=False
)
feature_effects.to_csv(
    output_dir / "model_coefficients.csv", index=False
)
pd.DataFrame([test_metrics]).to_csv(
    output_dir / "test_metrics.csv", index=False
)
threshold_summary.to_csv(
    output_dir / "validation_thresholds.csv", index=False
)

# Save preprocessing, model, threshold, and input information
model_bundle = {
    "model": final_model,
    "review_threshold": review_threshold,
    "feature_columns": feature_columns,
    "sklearn_version": sklearn.__version__,
    "data_source": "Synthetic credit-risk practice dataset",
    "input_requirement": "Cleaned features with missing values as np.nan"
}

model_path = output_dir / "credit_risk_model.joblib"
joblib.dump(model_bundle, model_path)

# Verify that reloading preserves predictions
loaded_bundle = joblib.load(model_path)
loaded_probabilities = loaded_bundle["model"].predict_proba(
    X_test
)[:, 1]

print("Saved outputs to:", output_dir.resolve())
print(
    "Reloaded predictions match:",
    np.allclose(test_probabilities, loaded_probabilities)
)

Saved outputs to: C:\Users\palre\Desktop\Interview_prep\data-science-interview\Credit Risk Project\credit_risk_outputs
Reloaded predictions match: True
